In [ ]:
# 1) Install dependencies, then restart runtime once
# Run this cell once to install dependencies and restart the runtime.
# After restart, run this cell again and then continue to the next cell.
import os
import subprocess
import sys
from pathlib import Path

MARKER = Path("/content/.snu_ai_challenge_pairwise_deps_installed")

if not MARKER.exists():
    packages = [
        "transformers>=4.49.0,<4.54.0",
        "accelerate>=0.34.0",
        "bitsandbytes>=0.46.1",
        "peft",
        "qwen-vl-utils",
        "jedi",
        "pandas==2.2.2",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *packages])
    MARKER.write_text("ok")
    print("Dependencies installed. Restarting runtime now. After restart, run this cell again, then continue.")
    os.kill(os.getpid(), 9)
else:
    print("Dependencies already installed. Continue to the next cell.")


In [ ]:
# 2) Setup + data unzip
from google.colab import drive

drive.mount('/content/drive')

import ast
import gc
import glob
import itertools
import json
import math
import os
import random
import re
import zipfile
from datetime import datetime

import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm
from torch.utils.data import Dataset, Subset
from transformers import AutoModelForVision2Seq, AutoProcessor, BitsAndBytesConfig, Trainer, TrainingArguments, set_seed
try:
    from transformers import Qwen2VLForConditionalGeneration
except ImportError:
    Qwen2VLForConditionalGeneration = AutoModelForVision2Seq
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training

ZIP_PATH = "/content/drive/MyDrive/SNU_AI_Challenge/snuaichallenge.zip"
DATA_DIR = "/content/snuaichallenge_data"
PAIRWISE_ROOT = "/content/drive/MyDrive/SNU_AI_Challenge/qwen2vl_pairwise_v1"

# Prompt variants:
# A_BASIC: images + sentence
# B_EVENTS: images + sentence + event stages
# C_DESCRIPTIONS: images + sentence + two frame descriptions
# D_FULL: images + sentence + event stages + two frame descriptions
PROMPT_VARIANT = "A_BASIC"
VARIANT_NAMES = {
    "A_BASIC": "A_basic",
    "B_EVENTS": "B_events",
    "C_DESCRIPTIONS": "C_descriptions",
    "D_FULL": "D_full",
}

RUN_ABLATION_ALL = True
RUN_VARIANTS = ["A_BASIC", "B_EVENTS", "C_DESCRIPTIONS", "D_FULL"]

CACHE_DIR = os.path.join(PAIRWISE_ROOT, "cache")
GENERATE_CACHES = True
CACHE_GENERATION_LIMIT = None
EVENT_CACHE_BATCH_SIZE = 8
FRAME_CACHE_BATCH_SIZE = 2

# Optional cache files for variants B/C/D. JSON format is expected.
EVENTS_CACHE_PATH = os.path.join(CACHE_DIR, "events_qwen2vl.json")
FRAME_DESCRIPTIONS_CACHE_PATH = os.path.join(CACHE_DIR, "frame_descriptions_qwen2vl.json")

# Set this to an existing variant run folder only when you want to skip training and evaluate that run.
# Example: "/content/drive/MyDrive/SNU_AI_Challenge/qwen2vl_pairwise_v1/runs/20260712_013000/A_basic"
EXISTING_OUTPUT_DIR = None

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
VARIANT_DIRNAME = VARIANT_NAMES[PROMPT_VARIANT]
RUN_ROOT = os.path.join(PAIRWISE_ROOT, "runs", RUN_ID)
OUTPUT_DIR = EXISTING_OUTPUT_DIR or os.path.join(RUN_ROOT, VARIANT_DIRNAME)
EVAL_DIR = os.path.join(OUTPUT_DIR, "eval")
SUBMIT_PATH = os.path.join(OUTPUT_DIR, "submission_pairwise.csv")

if not os.path.isdir(DATA_DIR):
    with zipfile.ZipFile(ZIP_PATH) as zip_file:
        zip_file.extractall("/content/")

TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
TEST_CSV = os.path.join(DATA_DIR, "test.csv")
TRAIN_IMAGE_DIR = os.path.join(DATA_DIR, "train")
TEST_IMAGE_DIR = os.path.join(DATA_DIR, "test")

for path in [PAIRWISE_ROOT, CACHE_DIR, OUTPUT_DIR, EVAL_DIR]:
    os.makedirs(path, exist_ok=True)

assert os.path.exists(TRAIN_CSV), TRAIN_CSV
assert os.path.exists(TEST_CSV), TEST_CSV
assert os.path.isdir(TRAIN_IMAGE_DIR), TRAIN_IMAGE_DIR
assert os.path.isdir(TEST_IMAGE_DIR), TEST_IMAGE_DIR

MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"
MIN_PIXELS = 128 * 28 * 28
MAX_PIXELS = 256 * 28 * 28
SEED = 42
VALID_RATIO = 0.1
SMOKE_TEST = False
ABLATION_TRAIN_ROWS = 300
ABLATION_VALID_ROWS = 100

# First-pass ablation budget. Use the same value for A/B/C/D.
# Set to None for a full epoch.
MAX_TRAIN_STEPS = 300

PAIR_INDICES = [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
PAIR_NAMES = ["1_vs_2", "1_vs_3", "1_vs_4", "2_vs_3", "2_vs_4", "3_vs_4"]

# Pairwise experiments use all 6 pairs per original sample.
TRAIN_PAIRS_PER_SAMPLE = len(PAIR_INDICES)
PAIR_BATCH_SIZE = 6
BEST_CHECKPOINT_EVAL_LIMIT = 100

def reset_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    set_seed(seed)


reset_all_seeds(SEED)

print("device:", "cuda" if torch.cuda.is_available() else "cpu")
print("data:", DATA_DIR)
print("pairwise root:", PAIRWISE_ROOT)
print("prompt variant:", PROMPT_VARIANT)
print("run output:", OUTPUT_DIR)


In [ ]:
# 3) Pairwise dataset, collator, scoring helpers
def parse_answer(answer):
    result = answer if isinstance(answer, list) else ast.literal_eval(str(answer))
    result = [int(value) for value in result]
    if len(result) != 4 or sorted(result) != [1, 2, 3, 4]:
        raise ValueError(f"Invalid Answer: {answer}")
    return result


train_df = pd.read_csv(TRAIN_CSV)
train_df["Id"] = train_df["Id"].astype(str)
train_df["Answer_list"] = train_df["Answer"].apply(parse_answer)

test_df = pd.read_csv(TEST_CSV)
test_df["Id"] = test_df["Id"].astype(str)

unique_ids = train_df["Id"].unique().copy()
rng = np.random.default_rng(SEED)
rng.shuffle(unique_ids)
valid_size = max(1, int(len(unique_ids) * VALID_RATIO))
valid_ids = set(unique_ids[:valid_size])
training_ids = set(unique_ids[valid_size:])

training_df = train_df[train_df["Id"].isin(training_ids)].reset_index(drop=True)
validation_df = train_df[train_df["Id"].isin(valid_ids)].reset_index(drop=True)

if ABLATION_TRAIN_ROWS is None:
    ablation_training_df = training_df.reset_index(drop=True)
else:
    ablation_training_df = training_df.sample(
        n=min(ABLATION_TRAIN_ROWS, len(training_df)),
        random_state=SEED,
    ).reset_index(drop=True)

if ABLATION_VALID_ROWS is None:
    ablation_validation_df = validation_df.reset_index(drop=True)
else:
    ablation_validation_df = validation_df.sample(
        n=min(ABLATION_VALID_ROWS, len(validation_df)),
        random_state=SEED,
    ).reset_index(drop=True)


def load_optional_json(path):
    if path is None or not os.path.exists(path):
        return {}
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


EVENTS_CACHE = load_optional_json(EVENTS_CACHE_PATH)
FRAME_DESCRIPTIONS_CACHE = load_optional_json(FRAME_DESCRIPTIONS_CACHE_PATH)

if PROMPT_VARIANT in {"B_EVENTS", "D_FULL"} and not EVENTS_CACHE:
    raise ValueError("PROMPT_VARIANT requires EVENTS_CACHE_PATH with cached sentence event stages.")

if PROMPT_VARIANT in {"C_DESCRIPTIONS", "D_FULL"} and not FRAME_DESCRIPTIONS_CACHE:
    raise ValueError("PROMPT_VARIANT requires FRAME_DESCRIPTIONS_CACHE_PATH with cached frame descriptions.")


def lookup_events(sample_id):
    value = EVENTS_CACHE.get(str(sample_id), [])
    if isinstance(value, dict):
        value = value.get("sentence_events", value.get("events", []))
    if isinstance(value, str):
        value = [value]
    return [str(item) for item in value]


def lookup_frame_description(sample_id, input_number):
    sample = FRAME_DESCRIPTIONS_CACHE.get(str(sample_id), {})
    key_options = [f"Input_{input_number}", str(input_number), input_number]
    value = ""
    for key in key_options:
        if key in sample:
            value = sample[key]
            break
    if isinstance(value, dict):
        return "\n".join(f"{key}: {val}" for key, val in value.items())
    return str(value)


def load_rgb(path):
    with Image.open(path) as image:
        return image.convert("RGB").copy()


def pair_target(answer_list, first_index, second_index):
    return "1" if int(answer_list[first_index]) < int(answer_list[second_index]) else "2"


def format_events(events):
    if not events:
        return ""
    lines = ["Event stages:"]
    for index, event in enumerate(events, start=1):
        lines.append(f"{index}. {event}")
    return "\n".join(lines)


def pair_instruction(sentence, sample_id=None, first_input=None, second_input=None):
    blocks = [f"Sentence:\n{sentence}"]

    if PROMPT_VARIANT in {"B_EVENTS", "D_FULL"}:
        blocks.append(format_events(lookup_events(sample_id)))

    if PROMPT_VARIANT in {"C_DESCRIPTIONS", "D_FULL"}:
        first_description = lookup_frame_description(sample_id, first_input)
        second_description = lookup_frame_description(sample_id, second_input)
        blocks.append(
            "First image evidence:\n"
            f"{first_description}\n\n"
            "Second image evidence:\n"
            f"{second_description}"
        )

    blocks.append(
        "The two images are different frames from the same video.\n"
        "Choose which image occurs earlier in the original timeline.\n"
        "If the first image is earlier, output 1.\n"
        "If the second image is earlier, output 2.\n"
        "Output only 1 or 2."
    )
    return "\n\n".join(block for block in blocks if block)


class PairwiseFrameDataset(Dataset):
    def __init__(self, dataframe, image_root, mode="train", pairs_per_sample=2, augment=True, seed=42):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_root = image_root
        self.mode = mode
        self.pairs_per_sample = pairs_per_sample
        self.augment = augment
        self.seed = seed
        self.records = []

        if mode not in {"train", "eval", "all_pairs"}:
            raise ValueError(f"Unsupported mode: {mode}")

        for row_index in range(len(self.dataframe)):
            if mode == "train":
                row_rng = random.Random(seed + row_index)
                selected_pairs = row_rng.sample(
                    PAIR_INDICES,
                    k=min(pairs_per_sample, len(PAIR_INDICES)),
                )
            else:
                selected_pairs = list(PAIR_INDICES)

            for pair_position, pair in enumerate(selected_pairs):
                self.records.append({
                    "row_index": row_index,
                    "pair_index": pair_position,
                    "pair": pair,
                })

    def __len__(self):
        return len(self.records)

    def _row_fields(self, row):
        sample_id = str(row["Id"])
        sentence = "" if pd.isna(row["Sentence"]) else str(row["Sentence"])
        image_paths = [
            os.path.join(self.image_root, sample_id, str(row[f"Input_{i}"]))
            for i in range(1, 5)
        ]
        answer_list = [int(value) for value in row["Answer_list"]]
        return sample_id, sentence, image_paths, answer_list

    def __getitem__(self, index):
        record = self.records[index]
        row_index = record["row_index"]
        item_index = record["pair_index"]
        row = self.dataframe.iloc[row_index]
        sample_id, sentence, image_paths, answer_list = self._row_fields(row)

        first_index, second_index = record["pair"]
        original_pair = tuple(sorted((first_index, second_index)))
        first_path = image_paths[first_index]
        second_path = image_paths[second_index]
        target = pair_target(answer_list, first_index, second_index)

        # Randomly reverse image presentation during training and flip the target.
        flipped = False
        if self.mode == "train" and self.augment:
            flip_rng = random.Random(self.seed * 100000 + index)
            if flip_rng.random() < 0.5:
                first_index, second_index = second_index, first_index
                first_path, second_path = second_path, first_path
                target = "1" if target == "2" else "2"
                flipped = True

        return {
            "Id": sample_id,
            "row_index": row_index,
            "pair_index": item_index,
            "pair_name": f"Input_{original_pair[0] + 1}_vs_Input_{original_pair[1] + 1}",
            "first_input": first_index + 1,
            "second_input": second_index + 1,
            "image_paths": [first_path, second_path],
            "image_labels": ["First image", "Second image"],
            "instruction": pair_instruction(
                sentence,
                sample_id=sample_id,
                first_input=first_index + 1,
                second_input=second_index + 1,
            ),
            "target": target,
            "answer_list": answer_list,
            "flipped": flipped,
        }


training_dataset = PairwiseFrameDataset(
    ablation_training_df,
    TRAIN_IMAGE_DIR,
    mode="train",
    pairs_per_sample=TRAIN_PAIRS_PER_SAMPLE,
    augment=True,
    seed=SEED,
)
validation_pair_dataset = PairwiseFrameDataset(
    ablation_validation_df,
    TRAIN_IMAGE_DIR,
    mode="eval",
    augment=False,
    seed=SEED,
)


def make_pairwise_messages(example):
    content = []
    for label in example["image_labels"]:
        content.append({"type": "text", "text": f"\n{label}:"})
        content.append({"type": "image"})
    content.append({"type": "text", "text": "\n\n" + example["instruction"]})
    return [{"role": "user", "content": content}]


def find_last_subsequence(sequence, pattern):
    for start in range(len(sequence) - len(pattern), -1, -1):
        if sequence[start:start + len(pattern)] == pattern:
            return start
    return -1


class QwenPairwiseCollator:
    def __init__(self, processor):
        self.processor = processor
        self.assistant_prefix_ids = processor.tokenizer.encode(
            "<|im_start|>assistant\n",
            add_special_tokens=False,
        )

    def __call__(self, examples):
        if len(examples) != 1:
            raise ValueError("Use batch size 1 with this collator.")

        example = examples[0]
        images = [load_rgb(path) for path in example["image_paths"]]
        messages = make_pairwise_messages(example) + [{"role": "assistant", "content": example["target"]}]
        text = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        model_inputs = self.processor(text=[text], images=images, padding=False, return_tensors="pt")

        input_ids = model_inputs["input_ids"][0].tolist()
        assistant_pos = find_last_subsequence(input_ids, self.assistant_prefix_ids)
        if assistant_pos >= 0:
            answer_start = assistant_pos + len(self.assistant_prefix_ids)
        else:
            prompt_text = self.processor.apply_chat_template(
                make_pairwise_messages(example),
                tokenize=False,
                add_generation_prompt=True,
            )
            prompt_inputs = self.processor(text=[prompt_text], images=images, padding=False, return_tensors="pt")
            answer_start = prompt_inputs["input_ids"].shape[1]

        labels = model_inputs["input_ids"].clone()
        labels[:, :answer_start] = -100
        if "attention_mask" in model_inputs:
            labels[model_inputs["attention_mask"] == 0] = -100
        model_inputs["labels"] = labels
        return model_inputs


def disable_sampling_warnings(model):
    generation_config = getattr(model, "generation_config", None)
    if generation_config is None:
        return
    generation_config.do_sample = False
    generation_config.temperature = None
    generation_config.top_p = None
    generation_config.top_k = None


def token_id_for_digit(digit):
    token_ids = processor.tokenizer.encode(str(digit), add_special_tokens=False)
    if len(token_ids) != 1:
        raise ValueError(f"Digit {digit} is not a single token: {token_ids}")
    return token_ids[0]


@torch.no_grad()
def score_pairwise_batch(examples):
    model.eval()
    disable_sampling_warnings(model)

    texts = []
    image_batches = []
    for example in examples:
        texts.append(
            processor.apply_chat_template(
                make_pairwise_messages(example),
                tokenize=False,
                add_generation_prompt=True,
            )
        )
        image_batches.append([load_rgb(path) for path in example["image_paths"]])

    inputs = processor(
        text=texts,
        images=image_batches,
        padding=True,
        return_tensors="pt",
    ).to(model.device)

    outputs = model(**inputs)
    attention_mask = inputs.get("attention_mask")
    if attention_mask is None:
        last_positions = torch.full(
            (outputs.logits.shape[0],),
            outputs.logits.shape[1] - 1,
            dtype=torch.long,
            device=outputs.logits.device,
        )
    else:
        attention_mask = attention_mask.to(outputs.logits.device)
        positions = torch.arange(
            attention_mask.shape[1],
            device=outputs.logits.device,
        ).unsqueeze(0).expand_as(attention_mask)
        last_positions = positions.masked_fill(attention_mask == 0, -1).max(dim=1).values

    batch_indices = torch.arange(outputs.logits.shape[0], device=outputs.logits.device)
    next_logits = outputs.logits[batch_indices, last_positions, :]

    one_id = token_id_for_digit("1")
    two_id = token_id_for_digit("2")
    pair_logits = next_logits[:, [one_id, two_id]].float()
    probs = torch.softmax(pair_logits, dim=-1).detach().cpu().numpy()

    results = []
    for prob in probs:
        p_first = float(prob[0])
        p_second = float(prob[1])
        results.append({
            "prediction": "1" if p_first >= p_second else "2",
            "p_first": p_first,
            "p_second": p_second,
            "confidence": max(p_first, p_second),
        })
    return results


@torch.no_grad()
def score_pairwise_example(example):
    return score_pairwise_batch([example])[0]


def order_list_to_rank_list(order):
    ranks = [0] * 4
    for position, input_index in enumerate(order, start=1):
        ranks[input_index] = position
    return ranks


def reconstruct_order_from_pair_probs(pair_prob_map):
    best_order = None
    best_score = -float("inf")
    eps = 1e-9
    for order in itertools.permutations(range(4)):
        score = 0.0
        for earlier_pos in range(4):
            for later_pos in range(earlier_pos + 1, 4):
                earlier = order[earlier_pos]
                later = order[later_pos]
                key = tuple(sorted((earlier, later)))
                p_first_earlier = pair_prob_map[key]
                if key[0] == earlier:
                    prob = p_first_earlier
                else:
                    prob = 1.0 - p_first_earlier
                score += math.log(max(prob, eps))
        if score > best_score:
            best_score = score
            best_order = order
    return list(best_order), best_score


def evaluate_pairwise_dataset(dataset, limit_rows=None, desc="Pairwise eval"):
    rows = []
    max_items = len(dataset)
    if limit_rows is not None:
        max_items = min(max_items, limit_rows * len(PAIR_INDICES))

    for start_index in tqdm(range(0, max_items, PAIR_BATCH_SIZE), desc=desc):
        end_index = min(start_index + PAIR_BATCH_SIZE, max_items)
        examples = [dataset[index] for index in range(start_index, end_index)]
        scored_batch = score_pairwise_batch(examples)

        for example, scored in zip(examples, scored_batch):
            rows.append({
                "Id": example["Id"],
                "row_index": example["row_index"],
                "pair_index": example["pair_index"],
                "pair_name": example["pair_name"],
                "first_input": example["first_input"],
                "second_input": example["second_input"],
                "target": example["target"],
                "prediction": scored["prediction"],
                "correct": scored["prediction"] == example["target"],
                "p_first": scored["p_first"],
                "p_second": scored["p_second"],
                "confidence": scored["confidence"],
                "answer_list": str(example["answer_list"]),
            })
    pair_df = pd.DataFrame(rows)
    order_df = reconstruct_orders_for_rows(pair_df)
    return pair_df, order_df


def reconstruct_orders_for_rows(pair_df):
    order_rows = []
    if pair_df.empty:
        return pd.DataFrame()

    for row_index, group in pair_df.groupby("row_index"):
        if len(group) != len(PAIR_INDICES):
            continue
        pair_prob_map = {}
        for _, row in group.iterrows():
            first_zero = int(row["first_input"]) - 1
            second_zero = int(row["second_input"]) - 1
            key = tuple(sorted((first_zero, second_zero)))
            if key[0] == first_zero:
                p_lower_index_earlier = float(row["p_first"])
            else:
                p_lower_index_earlier = float(row["p_second"])
            pair_prob_map[key] = p_lower_index_earlier

        best_order, score = reconstruct_order_from_pair_probs(pair_prob_map)
        prediction_ranks = order_list_to_rank_list(best_order)
        answer = parse_answer(group.iloc[0]["answer_list"])
        position_correct = sum(pred == target for pred, target in zip(prediction_ranks, answer))
        order_rows.append({
            "Id": group.iloc[0]["Id"],
            "row_index": int(row_index),
            "predicted_order_inputs": str([index + 1 for index in best_order]),
            "Prediction": str(prediction_ranks),
            "Answer": str(answer),
            "Position_correct": position_correct,
            "Position_total": 4,
            "Exact_correct": int(prediction_ranks == answer),
            "order_log_score": score,
            "mean_pair_confidence": float(group["confidence"].mean()),
        })
    return pd.DataFrame(order_rows)


def summarize_pairwise_metrics(pair_df, order_df):
    summary = {
        "pair_accuracy": float(pair_df["correct"].mean()) if not pair_df.empty else np.nan,
        "pair_correct": int(pair_df["correct"].sum()) if not pair_df.empty else 0,
        "pair_total": int(len(pair_df)),
        "position_accuracy": float(order_df["Position_correct"].sum() / order_df["Position_total"].sum()) if not order_df.empty else np.nan,
        "exact_match_accuracy": float(order_df["Exact_correct"].mean()) if not order_df.empty else np.nan,
        "exact_correct": int(order_df["Exact_correct"].sum()) if not order_df.empty else 0,
        "order_total": int(len(order_df)),
    }
    return summary


print("training pair examples:", len(training_dataset))
print("validation pair examples:", len(validation_pair_dataset))
print("training rows:", len(training_df), "validation rows:", len(validation_df), "test rows:", len(test_df))
print("ablation training rows:", len(ablation_training_df), "ablation validation rows:", len(ablation_validation_df))
print("example target:", training_dataset[0]["target"], training_dataset[0]["pair_name"])


In [ ]:
# This cell is kept for manual single-variant runs.
if RUN_ABLATION_ALL:
    print("Skipping single training cell because RUN_ABLATION_ALL is enabled. The ablation runner cell will handle it.")
else:
    # 4) Train pairwise LoRA
    reset_all_seeds(SEED)
    processor = AutoProcessor.from_pretrained(
        MODEL_ID,
        min_pixels=MIN_PIXELS,
        max_pixels=MAX_PIXELS,
    )

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )

    model = Qwen2VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        torch_dtype=torch.float16,
        device_map="auto",
    )
    model.config.use_cache = False

    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    model = get_peft_model(
        model,
        LoraConfig(
            r=8,
            lora_alpha=16,
            lora_dropout=0.05,
            bias="none",
            target_modules=["q_proj", "v_proj"],
            task_type="CAUSAL_LM",
        ),
    )

    for name, parameter in model.named_parameters():
        if "visual" in name.lower() or "vision" in name.lower():
            parameter.requires_grad = False

    trainable_parameter_names = [name for name, parameter in model.named_parameters() if parameter.requires_grad]
    print("Trainable parameter module count:", len(trainable_parameter_names))
    for name in trainable_parameter_names[:30]:
        print(name)
    model.print_trainable_parameters()

    data_collator = QwenPairwiseCollator(processor)

    if SMOKE_TEST:
        train_data = Subset(training_dataset, range(min(30, len(training_dataset))))
        max_steps = 100
    else:
        train_data = training_dataset
        max_steps = -1 if MAX_TRAIN_STEPS is None else MAX_TRAIN_STEPS
    warmup_steps = 60 if max_steps == -1 else max(1, int(max_steps * 0.05))

    training_argument_values = {
        "output_dir": OUTPUT_DIR,
        "num_train_epochs": 1,
        "max_steps": max_steps,
        "per_device_train_batch_size": 1,
        "gradient_accumulation_steps": 8,
        "learning_rate": 1e-4,
        "warmup_steps": warmup_steps,
        "max_grad_norm": 0.3,
        "fp16": True,
        "bf16": False,
        "gradient_checkpointing": True,
        "gradient_checkpointing_kwargs": {"use_reentrant": False},
        "optim": "paged_adamw_8bit",
        "logging_steps": 20,
        "save_strategy": "steps",
        "save_steps": 100,
        "save_total_limit": None,
        "report_to": "none",
        "remove_unused_columns": False,
        "dataloader_num_workers": 0,
        "seed": SEED,
        "data_seed": SEED,
    }

    if "eval_strategy" in TrainingArguments.__init__.__code__.co_varnames:
        training_argument_values["eval_strategy"] = "no"
    else:
        training_argument_values["evaluation_strategy"] = "no"

    training_args = TrainingArguments(**training_argument_values)

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_data,
        data_collator=data_collator,
    )

    trainer.train()
    trainer.save_model(OUTPUT_DIR)
    processor.save_pretrained(OUTPUT_DIR)

    run_config = {
        "run_id": RUN_ID,
        "output_dir": OUTPUT_DIR,
        "prompt_variant": PROMPT_VARIANT,
        "train_pairs_per_sample": TRAIN_PAIRS_PER_SAMPLE,
        "max_train_steps": MAX_TRAIN_STEPS,
        "events_cache_path": EVENTS_CACHE_PATH,
        "frame_descriptions_cache_path": FRAME_DESCRIPTIONS_CACHE_PATH,
        "seed": SEED,
        "valid_ratio": VALID_RATIO,
        "ablation_train_rows": len(ablation_training_df),
        "ablation_valid_rows": len(ablation_validation_df),
        "model_id": MODEL_ID,
    }
    with open(os.path.join(OUTPUT_DIR, "run_config.json"), "w", encoding="utf-8") as f:
        json.dump(run_config, f, ensure_ascii=False, indent=2)


In [ ]:
# This cell is kept for manual single-variant runs.
if RUN_ABLATION_ALL:
    print("Skipping single evaluation cell because RUN_ABLATION_ALL is enabled. The ablation runner cell will handle it.")
else:
    # 5) Eval checkpoints + choose best by pairwise-only exact-match
    if "processor" not in globals():
        processor = AutoProcessor.from_pretrained(
            MODEL_ID,
            min_pixels=MIN_PIXELS,
            max_pixels=MAX_PIXELS,
        )

    if "bnb_config" not in globals():
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16,
        )


    def checkpoint_step(path):
        base = os.path.basename(path.rstrip("/"))
        if base.startswith("checkpoint-"):
            return int(base.split("-")[-1])
        return 10**18


    def find_pairwise_adapter_dirs():
        checkpoints = sorted(
            glob.glob(os.path.join(OUTPUT_DIR, "checkpoint-*")),
            key=checkpoint_step,
        )
        candidates = [
            path for path in checkpoints
            if os.path.exists(os.path.join(path, "adapter_config.json"))
        ]
        if os.path.exists(os.path.join(OUTPUT_DIR, "adapter_config.json")):
            candidates.append(OUTPUT_DIR)
        return candidates


    def load_model_for_adapter(adapter_dir):
        base_model = Qwen2VLForConditionalGeneration.from_pretrained(
            MODEL_ID,
            quantization_config=bnb_config,
            torch_dtype=torch.float16,
            device_map="auto",
        )
        loaded_model = PeftModel.from_pretrained(base_model, adapter_dir)
        loaded_model.eval()
        loaded_model.config.use_cache = True
        disable_sampling_warnings(loaded_model)
        return loaded_model


    checkpoint_dirs = find_pairwise_adapter_dirs()
    print("Checkpoint count to evaluate:", len(checkpoint_dirs))

    if "model" not in globals():
        if not checkpoint_dirs:
            raise RuntimeError(f"No model in memory and no adapter checkpoints found in {OUTPUT_DIR}")
        print("No model in memory. Loading first checkpoint for initial eval:", checkpoint_dirs[0])
        model = load_model_for_adapter(checkpoint_dirs[0])

    model.eval()
    model.config.use_cache = True
    disable_sampling_warnings(model)

    current_pair_df, current_order_df = evaluate_pairwise_dataset(
        validation_pair_dataset,
        limit_rows=BEST_CHECKPOINT_EVAL_LIMIT,
        desc="Current model",
    )
    current_summary = summarize_pairwise_metrics(current_pair_df, current_order_df)
    print("Current summary:", current_summary)

    display(current_pair_df.head())
    display(current_order_df.head())

    pair_accuracy_by_name = current_pair_df.groupby("pair_name")["correct"].agg(["count", "mean"]).reset_index()
    display(pair_accuracy_by_name)

    confidence_bins = pd.cut(
        current_pair_df["confidence"],
        bins=[0.5, 0.6, 0.7, 0.8, 1.0],
        labels=["0.50-0.60", "0.60-0.70", "0.70-0.80", "0.80-1.00"],
        include_lowest=True,
    )
    confidence_summary = current_pair_df.assign(confidence_bin=confidence_bins).groupby("confidence_bin", observed=False)["correct"].agg(["count", "mean"]).reset_index()
    display(confidence_summary)

    current_pair_df.to_csv(os.path.join(EVAL_DIR, "current_pair_predictions.csv"), index=False)
    current_order_df.to_csv(os.path.join(EVAL_DIR, "current_reconstructed_orders.csv"), index=False)

    checkpoint_rows = []
    best_checkpoint_dir = None
    best_checkpoint_score = -1.0

    for checkpoint_dir in checkpoint_dirs:
        print("evaluating:", checkpoint_dir)
        if "model" in globals():
            del model
        gc.collect()
        torch.cuda.empty_cache()
        model = load_model_for_adapter(checkpoint_dir)

        pair_df, order_df = evaluate_pairwise_dataset(
            validation_pair_dataset,
            limit_rows=BEST_CHECKPOINT_EVAL_LIMIT,
            desc=os.path.basename(checkpoint_dir),
        )
        summary = summarize_pairwise_metrics(pair_df, order_df)
        summary["checkpoint"] = checkpoint_dir
        checkpoint_rows.append(summary)

        checkpoint_base = os.path.basename(checkpoint_dir.rstrip("/")) or "final"
        checkpoint_parent = os.path.basename(os.path.dirname(checkpoint_dir.rstrip("/")))
        if checkpoint_base.startswith("checkpoint-"):
            checkpoint_safe_name = f"{checkpoint_parent}_{checkpoint_base}"
        else:
            checkpoint_safe_name = checkpoint_base
        pair_df.to_csv(os.path.join(EVAL_DIR, f"{checkpoint_safe_name}_pair_predictions.csv"), index=False)
        order_df.to_csv(os.path.join(EVAL_DIR, f"{checkpoint_safe_name}_reconstructed_orders.csv"), index=False)

        score = summary["position_accuracy"]
        if score > best_checkpoint_score:
            best_checkpoint_score = score
            best_checkpoint_dir = checkpoint_dir

    if checkpoint_rows:
        checkpoint_result_df = pd.DataFrame(checkpoint_rows).sort_values(
            ["position_accuracy", "pair_accuracy", "exact_match_accuracy"],
            ascending=False,
        ).reset_index(drop=True)
        checkpoint_result_df.to_csv(os.path.join(EVAL_DIR, "checkpoint_summary.csv"), index=False)
        display(checkpoint_result_df)
        print("BEST_CHECKPOINT_DIR:", best_checkpoint_dir)
        print(f"BEST position accuracy: {best_checkpoint_score:.2%}")

        del model
        gc.collect()
        torch.cuda.empty_cache()
        model = load_model_for_adapter(best_checkpoint_dir)

        full_pair_df, full_order_df = evaluate_pairwise_dataset(
            validation_pair_dataset,
            limit_rows=None,
            desc="Best checkpoint full evaluation",
        )
        full_summary = summarize_pairwise_metrics(full_pair_df, full_order_df)
        print("Best checkpoint full summary:", full_summary)
        full_pair_df.to_csv(os.path.join(EVAL_DIR, "best_full_pair_predictions.csv"), index=False)
        full_order_df.to_csv(os.path.join(EVAL_DIR, "best_full_reconstructed_orders.csv"), index=False)
    else:
        checkpoint_result_df = pd.DataFrame()
        print("No adapter checkpoint found; keeping the current model.")


In [ ]:
# This cell is kept for manual single-variant runs.
if RUN_ABLATION_ALL:
    print("Skipping single inference cell because RUN_ABLATION_ALL is enabled. The ablation runner cell will handle it.")
else:
    # 6) Inference + submission from pairwise probabilities
    class PairwiseTestDataset(Dataset):
        def __init__(self, dataframe, image_root):
            self.dataframe = dataframe.reset_index(drop=True)
            self.image_root = image_root
            self.items_per_row = len(PAIR_INDICES)

        def __len__(self):
            return len(self.dataframe) * self.items_per_row

        def __getitem__(self, index):
            row_index = index // self.items_per_row
            pair_index = index % self.items_per_row
            row = self.dataframe.iloc[row_index]
            sample_id = str(row["Id"])
            sentence = "" if pd.isna(row["Sentence"]) else str(row["Sentence"])
            first_index, second_index = PAIR_INDICES[pair_index]
            image_paths = [
                os.path.join(self.image_root, sample_id, str(row[f"Input_{first_index + 1}"])),
                os.path.join(self.image_root, sample_id, str(row[f"Input_{second_index + 1}"])),
            ]
            return {
                "Id": sample_id,
                "row_index": row_index,
                "pair_index": pair_index,
                "pair_name": f"Input_{first_index + 1}_vs_Input_{second_index + 1}",
                "first_input": first_index + 1,
                "second_input": second_index + 1,
                "image_paths": image_paths,
                "image_labels": ["First image", "Second image"],
                "instruction": pair_instruction(
                    sentence,
                    sample_id=sample_id,
                    first_input=first_index + 1,
                    second_input=second_index + 1,
                ),
            }


    test_pair_dataset = PairwiseTestDataset(test_df, TEST_IMAGE_DIR)

    test_pair_rows = []
    for start_index in tqdm(range(0, len(test_pair_dataset), PAIR_BATCH_SIZE), desc="Pairwise inference"):
        end_index = min(start_index + PAIR_BATCH_SIZE, len(test_pair_dataset))
        examples = [test_pair_dataset[index] for index in range(start_index, end_index)]
        scored_batch = score_pairwise_batch(examples)
        for example, scored in zip(examples, scored_batch):
            test_pair_rows.append({
                "Id": example["Id"],
                "row_index": example["row_index"],
                "pair_index": example["pair_index"],
                "pair_name": example["pair_name"],
                "first_input": example["first_input"],
                "second_input": example["second_input"],
                "prediction": scored["prediction"],
                "p_first": scored["p_first"],
                "p_second": scored["p_second"],
                "confidence": scored["confidence"],
            })

    test_pair_df = pd.DataFrame(test_pair_rows)

    submission_rows = []
    for row_index, group in test_pair_df.groupby("row_index"):
        if len(group) != len(PAIR_INDICES):
            raise RuntimeError(f"Missing pair predictions for row_index={row_index}")
        pair_prob_map = {}
        for _, row in group.iterrows():
            first_zero = int(row["first_input"]) - 1
            second_zero = int(row["second_input"]) - 1
            key = tuple(sorted((first_zero, second_zero)))
            if key[0] == first_zero:
                p_lower_index_earlier = float(row["p_first"])
            else:
                p_lower_index_earlier = float(row["p_second"])
            pair_prob_map[key] = p_lower_index_earlier

        best_order, score = reconstruct_order_from_pair_probs(pair_prob_map)
        prediction_ranks = order_list_to_rank_list(best_order)
        sample_id = group.iloc[0]["Id"]
        submission_rows.append({
            "Id": sample_id,
            "Answer": str(prediction_ranks),
            "predicted_order_inputs": str([index + 1 for index in best_order]),
            "order_log_score": score,
            "mean_pair_confidence": float(group["confidence"].mean()),
        })

    submission_debug_df = pd.DataFrame(submission_rows)
    submission_df = submission_debug_df[["Id", "Answer"]].copy()

    pair_prediction_path = os.path.join(OUTPUT_DIR, "test_pair_predictions.csv")
    submission_debug_path = os.path.join(OUTPUT_DIR, "submission_pairwise_debug.csv")

    test_pair_df.to_csv(pair_prediction_path, index=False)
    submission_debug_df.to_csv(submission_debug_path, index=False)
    submission_df.to_csv(SUBMIT_PATH, index=False)

    print("saved submission:", SUBMIT_PATH)
    print("saved pair predictions:", pair_prediction_path)
    print("saved debug submission:", submission_debug_path)
    if "best_checkpoint_dir" in globals():
        print("using checkpoint:", best_checkpoint_dir)
    display(submission_df.head())


In [ ]:
# 7) Generate event/frame-description caches for B/C/D variants
# This cell runs only when RUN_ABLATION_ALL and GENERATE_CACHES are True.
if not RUN_ABLATION_ALL:
    print("Skipping cache generation because RUN_ABLATION_ALL is disabled.")
elif not GENERATE_CACHES:
    print("Skipping cache generation because GENERATE_CACHES is False.")
else:
    os.makedirs(CACHE_DIR, exist_ok=True)

    if "processor" not in globals():
        processor = AutoProcessor.from_pretrained(
            MODEL_ID,
            min_pixels=MIN_PIXELS,
            max_pixels=MAX_PIXELS,
        )

    if "bnb_config" not in globals():
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16,
        )

    def load_cache_generation_model():
        cache_model = Qwen2VLForConditionalGeneration.from_pretrained(
            MODEL_ID,
            quantization_config=bnb_config,
            torch_dtype=torch.float16,
            device_map="auto",
        )
        cache_model.eval()
        cache_model.config.use_cache = True
        disable_sampling_warnings(cache_model)
        return cache_model

    @torch.no_grad()
    def generate_text_batch(cache_model, messages_batch, images_batch=None, max_new_tokens=160):
        texts = [
            processor.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )
            for messages in messages_batch
        ]
        kwargs = {
            "text": texts,
            "padding": True,
            "return_tensors": "pt",
        }
        if images_batch is not None:
            kwargs["images"] = images_batch
        inputs = processor(**kwargs).to(cache_model.device)

        generated_ids = cache_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
        generated_ids_trimmed = [
            out_ids[len(in_ids):]
            for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        return [
            text.strip()
            for text in processor.batch_decode(
                generated_ids_trimmed,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=False,
            )
        ]

    def parse_event_lines(text):
        lines = []
        for raw_line in text.splitlines():
            line = raw_line.strip()
            line = re.sub(r"^[-*\d\.\)\s]+", "", line).strip()
            if line:
                lines.append(line)
        return lines[:6] if lines else [text.strip()]

    def rows_with_image_root(dataframe, image_root):
        rows = dataframe.copy()
        rows["__image_root"] = image_root
        rows["Id"] = rows["Id"].astype(str)
        return rows

    def ablation_rows_for_cache():
        combined = pd.concat(
            [
                rows_with_image_root(ablation_training_df, TRAIN_IMAGE_DIR),
                rows_with_image_root(ablation_validation_df, TRAIN_IMAGE_DIR),
            ],
            ignore_index=True,
            sort=False,
        )
        return combined.drop_duplicates("Id").reset_index(drop=True)

    def test_rows_for_cache():
        return rows_with_image_root(test_df, TEST_IMAGE_DIR).drop_duplicates("Id").reset_index(drop=True)

    def chunked(items, chunk_size):
        for start in range(0, len(items), chunk_size):
            yield items[start:start + chunk_size]

    def generate_cache_for_rows(cache_model, cache_rows, need_events=True, need_frame_descriptions=True):
        global EVENTS_CACHE, FRAME_DESCRIPTIONS_CACHE
        EVENTS_CACHE = load_optional_json(EVENTS_CACHE_PATH)
        FRAME_DESCRIPTIONS_CACHE = load_optional_json(FRAME_DESCRIPTIONS_CACHE_PATH)

        event_jobs = []
        if need_events:
            for _, row in cache_rows.iterrows():
                sample_id = str(row["Id"])
                if sample_id in EVENTS_CACHE:
                    continue
                sentence = "" if pd.isna(row.get("Sentence")) else str(row.get("Sentence"))
                prompt = (
                    "Decompose the sentence into 1 to 6 chronological event stages.\n"
                    "Write only directly implied events. Do not invent intermediate actions.\n"
                    "If no clear multi-stage sequence exists, return one event only.\n"
                    "Return one short event per line.\n\n"
                    f"Sentence:\n{sentence}"
                )
                messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
                event_jobs.append((sample_id, messages))

            for batch in tqdm(list(chunked(event_jobs, EVENT_CACHE_BATCH_SIZE)), desc="Generating event cache"):
                outputs = generate_text_batch(
                    cache_model,
                    [messages for _, messages in batch],
                    images_batch=None,
                    max_new_tokens=160,
                )
                for (sample_id, _), output in zip(batch, outputs):
                    EVENTS_CACHE[sample_id] = parse_event_lines(output)
                with open(EVENTS_CACHE_PATH, "w", encoding="utf-8") as f:
                    json.dump(EVENTS_CACHE, f, ensure_ascii=False, indent=2)

        frame_jobs = []
        if need_frame_descriptions:
            for _, row in cache_rows.iterrows():
                sample_id = str(row["Id"])
                if sample_id not in FRAME_DESCRIPTIONS_CACHE:
                    FRAME_DESCRIPTIONS_CACHE[sample_id] = {}
                image_root = row["__image_root"]
                for input_number in range(1, 5):
                    input_key = f"Input_{input_number}"
                    if input_key in FRAME_DESCRIPTIONS_CACHE[sample_id]:
                        continue
                    if input_key not in row or pd.isna(row[input_key]):
                        continue
                    image_path = os.path.join(image_root, sample_id, str(row[input_key]))
                    prompt = (
                        "Describe only directly visible facts in this frame.\n"
                        "Do not infer temporal order.\n"
                        "Do not mention whether the frame is earlier or later.\n"
                        "Do not use or refer to the sentence.\n"
                        "Use compact structured lines for:\n"
                        "- person\n"
                        "- background\n"
                        "- current_action\n"
                        "- object_state\n"
                        "- body_and_hand_position"
                    )
                    messages = [{
                        "role": "user",
                        "content": [
                            {"type": "image"},
                            {"type": "text", "text": prompt},
                        ],
                    }]
                    frame_jobs.append((sample_id, input_key, image_path, messages))

            for batch in tqdm(list(chunked(frame_jobs, FRAME_CACHE_BATCH_SIZE)), desc="Generating frame-description cache"):
                outputs = generate_text_batch(
                    cache_model,
                    [messages for _, _, _, messages in batch],
                    images_batch=[[load_rgb(image_path)] for _, _, image_path, _ in batch],
                    max_new_tokens=180,
                )
                for (sample_id, input_key, _, _), output in zip(batch, outputs):
                    if sample_id not in FRAME_DESCRIPTIONS_CACHE:
                        FRAME_DESCRIPTIONS_CACHE[sample_id] = {}
                    FRAME_DESCRIPTIONS_CACHE[sample_id][input_key] = output
                with open(FRAME_DESCRIPTIONS_CACHE_PATH, "w", encoding="utf-8") as f:
                    json.dump(FRAME_DESCRIPTIONS_CACHE, f, ensure_ascii=False, indent=2)

    cache_rows = ablation_rows_for_cache()
    if CACHE_GENERATION_LIMIT is not None:
        cache_rows = cache_rows.head(CACHE_GENERATION_LIMIT).reset_index(drop=True)

    cache_model = load_cache_generation_model()

    try:
        generate_cache_for_rows(
            cache_model,
            cache_rows,
            need_events=any(variant in {"B_EVENTS", "D_FULL"} for variant in RUN_VARIANTS),
            need_frame_descriptions=any(variant in {"C_DESCRIPTIONS", "D_FULL"} for variant in RUN_VARIANTS),
        )

    finally:
        del cache_model
        gc.collect()
        torch.cuda.empty_cache()

    print("event cache:", EVENTS_CACHE_PATH, len(EVENTS_CACHE))
    print("frame description cache:", FRAME_DESCRIPTIONS_CACHE_PATH, len(FRAME_DESCRIPTIONS_CACHE))


In [ ]:
# 8) Run A/B/C/D ablation end-to-end
if not RUN_ABLATION_ALL:
    print("RUN_ABLATION_ALL is disabled. Use cells 4-6 for a manual single-variant run.")
else:
    ablation_rows = []

    def set_variant_paths(variant):
        global PROMPT_VARIANT, OUTPUT_DIR, EVAL_DIR, SUBMIT_PATH
        PROMPT_VARIANT = variant
        variant_dirname = VARIANT_NAMES[variant]
        OUTPUT_DIR = EXISTING_OUTPUT_DIR or os.path.join(RUN_ROOT, variant_dirname)
        EVAL_DIR = os.path.join(OUTPUT_DIR, "eval")
        SUBMIT_PATH = os.path.join(OUTPUT_DIR, "submission_pairwise.csv")
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        os.makedirs(EVAL_DIR, exist_ok=True)
        print("=" * 80)
        print("variant:", variant, "output:", OUTPUT_DIR)

    def validate_variant_inputs(variant, rows=None):
        global EVENTS_CACHE, FRAME_DESCRIPTIONS_CACHE
        EVENTS_CACHE = load_optional_json(EVENTS_CACHE_PATH)
        FRAME_DESCRIPTIONS_CACHE = load_optional_json(FRAME_DESCRIPTIONS_CACHE_PATH)
        if variant in {"B_EVENTS", "D_FULL"} and not EVENTS_CACHE:
            raise RuntimeError(f"{variant} requires EVENTS_CACHE_PATH: {EVENTS_CACHE_PATH}")
        if variant in {"C_DESCRIPTIONS", "D_FULL"} and not FRAME_DESCRIPTIONS_CACHE:
            raise RuntimeError(f"{variant} requires FRAME_DESCRIPTIONS_CACHE_PATH: {FRAME_DESCRIPTIONS_CACHE_PATH}")
        if rows is None:
            rows = pd.concat([ablation_training_df, ablation_validation_df], ignore_index=True, sort=False)
        rows = rows.reset_index(drop=True)
        if variant in {"B_EVENTS", "D_FULL"}:
            missing_event_ids = [
                str(row["Id"]) for _, row in rows.iterrows()
                if str(row["Id"]) not in EVENTS_CACHE
            ]
            if missing_event_ids:
                raise RuntimeError(
                    f"{variant} event cache is missing {len(missing_event_ids)} sample(s). "
                    f"First missing ids: {missing_event_ids[:10]}"
                )
        if variant in {"C_DESCRIPTIONS", "D_FULL"}:
            missing_frame_keys = []
            for _, row in rows.iterrows():
                sample_id = str(row["Id"])
                sample_cache = FRAME_DESCRIPTIONS_CACHE.get(sample_id, {})
                for input_number in range(1, 5):
                    input_key = f"Input_{input_number}"
                    if input_key in row and not pd.isna(row[input_key]) and input_key not in sample_cache:
                        missing_frame_keys.append(f"{sample_id}:{input_key}")
            if missing_frame_keys:
                raise RuntimeError(
                    f"{variant} frame-description cache is missing {len(missing_frame_keys)} frame(s). "
                    f"First missing keys: {missing_frame_keys[:10]}"
                )

    def build_bnb_config():
        return BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16,
        )

    def load_trainable_pairwise_model():
        loaded_model = Qwen2VLForConditionalGeneration.from_pretrained(
            MODEL_ID,
            quantization_config=bnb_config,
            torch_dtype=torch.float16,
            device_map="auto",
        )
        loaded_model.config.use_cache = False
        loaded_model = prepare_model_for_kbit_training(
            loaded_model,
            use_gradient_checkpointing=True,
        )
        loaded_model = get_peft_model(
            loaded_model,
            LoraConfig(
                r=8,
                lora_alpha=16,
                lora_dropout=0.05,
                bias="none",
                target_modules=["q_proj", "v_proj"],
                task_type="CAUSAL_LM",
            ),
        )
        for name, parameter in loaded_model.named_parameters():
            if "visual" in name.lower() or "vision" in name.lower():
                parameter.requires_grad = False
        return loaded_model

    def checkpoint_step(path):
        base = os.path.basename(path.rstrip("/"))
        if base.startswith("checkpoint-"):
            return int(base.split("-")[-1])
        return 10**18

    def find_pairwise_adapter_dirs_for_current_output():
        checkpoints = sorted(
            glob.glob(os.path.join(OUTPUT_DIR, "checkpoint-*")),
            key=checkpoint_step,
        )
        candidates = [
            path for path in checkpoints
            if os.path.exists(os.path.join(path, "adapter_config.json"))
        ]
        if os.path.exists(os.path.join(OUTPUT_DIR, "adapter_config.json")):
            candidates.append(OUTPUT_DIR)
        return candidates

    def load_model_for_adapter(adapter_dir):
        base_model = Qwen2VLForConditionalGeneration.from_pretrained(
            MODEL_ID,
            quantization_config=bnb_config,
            torch_dtype=torch.float16,
            device_map="auto",
        )
        loaded_model = PeftModel.from_pretrained(base_model, adapter_dir)
        loaded_model.eval()
        loaded_model.config.use_cache = True
        disable_sampling_warnings(loaded_model)
        return loaded_model

    def train_current_variant():
        global model
        reset_all_seeds(SEED)
        model = load_trainable_pairwise_model()
        trainable_parameter_names = [
            name for name, parameter in model.named_parameters()
            if parameter.requires_grad
        ]
        print("Trainable parameter module count:", len(trainable_parameter_names))
        model.print_trainable_parameters()

        data_collator = QwenPairwiseCollator(processor)
        if SMOKE_TEST:
            train_data = Subset(training_dataset, range(min(30, len(training_dataset))))
            max_steps = 100
        else:
            train_data = training_dataset
            max_steps = -1 if MAX_TRAIN_STEPS is None else MAX_TRAIN_STEPS
        warmup_steps = 60 if max_steps == -1 else max(1, int(max_steps * 0.05))

        training_argument_values = {
            "output_dir": OUTPUT_DIR,
            "num_train_epochs": 1,
            "max_steps": max_steps,
            "per_device_train_batch_size": 1,
            "gradient_accumulation_steps": 8,
            "learning_rate": 1e-4,
            "warmup_steps": warmup_steps,
            "max_grad_norm": 0.3,
            "fp16": True,
            "bf16": False,
            "gradient_checkpointing": True,
            "gradient_checkpointing_kwargs": {"use_reentrant": False},
            "optim": "paged_adamw_8bit",
            "logging_steps": 20,
            "save_strategy": "steps",
            "save_steps": 100,
            "save_total_limit": None,
            "report_to": "none",
            "remove_unused_columns": False,
            "dataloader_num_workers": 0,
            "seed": SEED,
            "data_seed": SEED,
        }
        if "eval_strategy" in TrainingArguments.__init__.__code__.co_varnames:
            training_argument_values["eval_strategy"] = "no"
        else:
            training_argument_values["evaluation_strategy"] = "no"

        trainer = Trainer(
            model=model,
            args=TrainingArguments(**training_argument_values),
            train_dataset=train_data,
            data_collator=data_collator,
        )
        trainer.train()
        trainer.save_model(OUTPUT_DIR)
        processor.save_pretrained(OUTPUT_DIR)

        run_config = {
            "run_id": RUN_ID,
            "output_dir": OUTPUT_DIR,
            "prompt_variant": PROMPT_VARIANT,
            "train_pairs_per_sample": TRAIN_PAIRS_PER_SAMPLE,
            "max_train_steps": MAX_TRAIN_STEPS,
            "events_cache_path": EVENTS_CACHE_PATH,
            "frame_descriptions_cache_path": FRAME_DESCRIPTIONS_CACHE_PATH,
            "seed": SEED,
            "valid_ratio": VALID_RATIO,
            "ablation_train_rows": len(ablation_training_df),
            "ablation_valid_rows": len(ablation_validation_df),
            "model_id": MODEL_ID,
        }
        with open(os.path.join(OUTPUT_DIR, "run_config.json"), "w", encoding="utf-8") as f:
            json.dump(run_config, f, ensure_ascii=False, indent=2)

    def evaluate_current_variant():
        global model
        checkpoint_dirs = find_pairwise_adapter_dirs_for_current_output()
        if not checkpoint_dirs:
            raise RuntimeError(f"No adapter checkpoints found in {OUTPUT_DIR}")

        checkpoint_rows = []
        best_checkpoint_dir = None
        best_checkpoint_score = -1.0

        for checkpoint_dir in checkpoint_dirs:
            print("evaluating:", checkpoint_dir)
            if "model" in globals():
                del model
            gc.collect()
            torch.cuda.empty_cache()
            model = load_model_for_adapter(checkpoint_dir)
            pair_df, order_df = evaluate_pairwise_dataset(
                validation_pair_dataset,
                limit_rows=BEST_CHECKPOINT_EVAL_LIMIT,
                desc=f"{PROMPT_VARIANT}:{os.path.basename(checkpoint_dir)}",
            )
            summary = summarize_pairwise_metrics(pair_df, order_df)
            summary["checkpoint"] = checkpoint_dir
            checkpoint_rows.append(summary)

            checkpoint_base = os.path.basename(checkpoint_dir.rstrip("/")) or "final"
            pair_df.to_csv(os.path.join(EVAL_DIR, f"{checkpoint_base}_pair_predictions.csv"), index=False)
            order_df.to_csv(os.path.join(EVAL_DIR, f"{checkpoint_base}_reconstructed_orders.csv"), index=False)

            score = summary["position_accuracy"]
            if score > best_checkpoint_score:
                best_checkpoint_score = score
                best_checkpoint_dir = checkpoint_dir

        checkpoint_result_df = pd.DataFrame(checkpoint_rows).sort_values(
            ["position_accuracy", "pair_accuracy", "exact_match_accuracy"],
            ascending=False,
        ).reset_index(drop=True)
        checkpoint_result_df.to_csv(os.path.join(EVAL_DIR, "checkpoint_summary.csv"), index=False)
        display(checkpoint_result_df)

        if "model" in globals():
            del model
        gc.collect()
        torch.cuda.empty_cache()
        model = load_model_for_adapter(best_checkpoint_dir)

        full_pair_df, full_order_df = evaluate_pairwise_dataset(
            validation_pair_dataset,
            limit_rows=None,
            desc=f"{PROMPT_VARIANT}:best full eval",
        )
        full_summary = summarize_pairwise_metrics(full_pair_df, full_order_df)
        full_summary["variant"] = PROMPT_VARIANT
        full_summary["best_checkpoint"] = best_checkpoint_dir
        full_pair_df.to_csv(os.path.join(EVAL_DIR, "best_full_pair_predictions.csv"), index=False)
        full_order_df.to_csv(os.path.join(EVAL_DIR, "best_full_reconstructed_orders.csv"), index=False)
        return full_summary, best_checkpoint_dir

    class PairwiseTestDataset(Dataset):
        def __init__(self, dataframe, image_root):
            self.dataframe = dataframe.reset_index(drop=True)
            self.image_root = image_root
            self.items_per_row = len(PAIR_INDICES)

        def __len__(self):
            return len(self.dataframe) * self.items_per_row

        def __getitem__(self, index):
            row_index = index // self.items_per_row
            pair_index = index % self.items_per_row
            row = self.dataframe.iloc[row_index]
            sample_id = str(row["Id"])
            sentence = "" if pd.isna(row["Sentence"]) else str(row["Sentence"])
            first_index, second_index = PAIR_INDICES[pair_index]
            image_paths = [
                os.path.join(self.image_root, sample_id, str(row[f"Input_{first_index + 1}"])),
                os.path.join(self.image_root, sample_id, str(row[f"Input_{second_index + 1}"])),
            ]
            return {
                "Id": sample_id,
                "row_index": row_index,
                "pair_index": pair_index,
                "pair_name": f"Input_{first_index + 1}_vs_Input_{second_index + 1}",
                "first_input": first_index + 1,
                "second_input": second_index + 1,
                "image_paths": image_paths,
                "image_labels": ["First image", "Second image"],
                "instruction": pair_instruction(
                    sentence,
                    sample_id=sample_id,
                    first_input=first_index + 1,
                    second_input=second_index + 1,
                ),
            }

    def run_inference_current_variant(best_checkpoint_dir):
        global model
        if "model" in globals():
            del model
        gc.collect()
        torch.cuda.empty_cache()
        model = load_model_for_adapter(best_checkpoint_dir)

        test_pair_dataset = PairwiseTestDataset(test_df, TEST_IMAGE_DIR)
        test_pair_rows = []
        for start_index in tqdm(range(0, len(test_pair_dataset), PAIR_BATCH_SIZE), desc=f"{PROMPT_VARIANT}:inference"):
            end_index = min(start_index + PAIR_BATCH_SIZE, len(test_pair_dataset))
            examples = [test_pair_dataset[index] for index in range(start_index, end_index)]
            scored_batch = score_pairwise_batch(examples)
            for example, scored in zip(examples, scored_batch):
                test_pair_rows.append({
                    "Id": example["Id"],
                    "row_index": example["row_index"],
                    "pair_index": example["pair_index"],
                    "pair_name": example["pair_name"],
                    "first_input": example["first_input"],
                    "second_input": example["second_input"],
                    "prediction": scored["prediction"],
                    "p_first": scored["p_first"],
                    "p_second": scored["p_second"],
                    "confidence": scored["confidence"],
                })

        test_pair_df = pd.DataFrame(test_pair_rows)
        submission_rows = []
        for row_index, group in test_pair_df.groupby("row_index"):
            pair_prob_map = {}
            for _, row in group.iterrows():
                first_zero = int(row["first_input"]) - 1
                second_zero = int(row["second_input"]) - 1
                key = tuple(sorted((first_zero, second_zero)))
                if key[0] == first_zero:
                    p_lower_index_earlier = float(row["p_first"])
                else:
                    p_lower_index_earlier = float(row["p_second"])
                pair_prob_map[key] = p_lower_index_earlier
            best_order, score = reconstruct_order_from_pair_probs(pair_prob_map)
            prediction_ranks = order_list_to_rank_list(best_order)
            submission_rows.append({
                "Id": group.iloc[0]["Id"],
                "Answer": str(prediction_ranks),
                "predicted_order_inputs": str([index + 1 for index in best_order]),
                "order_log_score": score,
                "mean_pair_confidence": float(group["confidence"].mean()),
            })

        submission_debug_df = pd.DataFrame(submission_rows)
        submission_df = submission_debug_df[["Id", "Answer"]].copy()
        test_pair_df.to_csv(os.path.join(OUTPUT_DIR, "test_pair_predictions.csv"), index=False)
        submission_debug_df.to_csv(os.path.join(OUTPUT_DIR, "submission_pairwise_debug.csv"), index=False)
        submission_df.to_csv(SUBMIT_PATH, index=False)
        print("saved submission:", SUBMIT_PATH)

    def ensure_test_cache_for_variant(variant):
        if variant not in {"B_EVENTS", "C_DESCRIPTIONS", "D_FULL"}:
            return
        if not GENERATE_CACHES:
            validate_variant_inputs(variant)
            return
        need_events = variant in {"B_EVENTS", "D_FULL"}
        need_frame_descriptions = variant in {"C_DESCRIPTIONS", "D_FULL"}
        missing_helpers = [
            name for name in [
                "load_cache_generation_model",
                "test_rows_for_cache",
                "generate_cache_for_rows",
            ]
            if name not in globals()
        ]
        if missing_helpers:
            raise RuntimeError(
                "Cache helper functions are missing. Run the cache generation cell before the ablation runner. "
                f"Missing: {missing_helpers}"
            )

        print("Generating test cache only for best variant:", variant)
        cache_model = load_cache_generation_model()
        try:
            generate_cache_for_rows(
                cache_model,
                test_rows_for_cache(),
                need_events=need_events,
                need_frame_descriptions=need_frame_descriptions,
            )
        finally:
            del cache_model
            gc.collect()
            torch.cuda.empty_cache()
        validate_variant_inputs(variant, rows=test_df)

    if "processor" not in globals():
        processor = AutoProcessor.from_pretrained(
            MODEL_ID,
            min_pixels=MIN_PIXELS,
            max_pixels=MAX_PIXELS,
        )
    bnb_config = build_bnb_config()

    for variant in RUN_VARIANTS:
        set_variant_paths(variant)
        validate_variant_inputs(variant)
        train_current_variant()
        full_summary, best_checkpoint_dir = evaluate_current_variant()
        ablation_rows.append(full_summary)
        if "model" in globals():
            del model
        gc.collect()
        torch.cuda.empty_cache()

    ablation_summary_df = pd.DataFrame(ablation_rows).sort_values(
        ["position_accuracy", "pair_accuracy", "exact_match_accuracy"],
        ascending=False,
    ).reset_index(drop=True)
    ablation_summary_path = os.path.join(RUN_ROOT, "ablation_summary.csv")
    ablation_summary_df.to_csv(ablation_summary_path, index=False)
    print("saved ablation summary:", ablation_summary_path)
    display(ablation_summary_df)

    best_row = ablation_summary_df.iloc[0]
    best_variant = best_row["variant"]
    best_checkpoint_dir = best_row["best_checkpoint"]
    set_variant_paths(best_variant)
    ensure_test_cache_for_variant(best_variant)
    run_inference_current_variant(best_checkpoint_dir)
